In [1]:
pip install -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 5.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:


from pathlib import Path

import yaml
from ultralytics import RTDETR

# =========================================================
# CONFIG — edit these values for your setup
# =========================================================
ORIGINAL_DATA_YAML = "/kaggle/input/datasets/nomanhossain4/green-dataset/Vehicles.v5i.yolov11/data.yaml"
# ^ path to the dataset's ORIGINAL data.yaml (read-only is fine — we never write here)

WRITABLE_DIR = "/kaggle/working"   # a folder you can write to (Kaggle output dir)

MODEL_SIZE  = "rtdetr-l.pt"        # RT-DETR only ships 'rtdetr-l.pt' and 'rtdetr-x.pt'
EPOCHS      = 20
IMG_SIZE    = 640
BATCH       = 16
DEVICE      = 0                    # GPU index (0) or "cpu"
PATIENCE    = 30                   # early stopping (epochs with no val improvement)
AMP         = False                # RT-DETR's deformable attention can produce NaNs under AMP


def build_resolved_data_yaml():
    """
    Read the ORIGINAL data.yaml (whatever relative paths it contains) and
    write a NEW data.yaml with fully absolute train/val/test paths, based on
    where the images actually sit on disk (siblings of the original yaml —
    i.e. <dataset_root>/train/images, <dataset_root>/valid/images,
    <dataset_root>/test/images).
    """
    orig_path = Path(ORIGINAL_DATA_YAML).resolve()
    with open(orig_path) as f:
        cfg = yaml.safe_load(f)

    dataset_root = orig_path.parent  # folder that actually contains train/valid/test

    resolved = {
        "train": str((dataset_root / "train" / "images").resolve()),
        "val":   str((dataset_root / "valid" / "images").resolve()),
        "test":  str((dataset_root / "test" / "images").resolve()),
        "nc":    cfg["nc"],
        "names": cfg["names"],
    }

    for split, img_dir in [("train", resolved["train"]), ("val", resolved["val"]), ("test", resolved["test"])]:
        if not Path(img_dir).exists():
            raise FileNotFoundError(f"'{split}' images dir does not exist: {img_dir}")

    Path(WRITABLE_DIR).mkdir(parents=True, exist_ok=True)
    new_yaml_path = Path(WRITABLE_DIR) / "data_resolved.yaml"
    with open(new_yaml_path, "w") as f:
        yaml.safe_dump(resolved, f, sort_keys=False)

    print(f"Resolved data.yaml written to: {new_yaml_path}")
    for k in ("train", "val", "test"):
        print(f"  {k}: {resolved[k]}")

    return str(new_yaml_path)


def print_metrics(metrics, label):
    print(f"\n--- {label} ---")
    print(f"Precision: {metrics.box.mp:.4f}")
    print(f"Recall:    {metrics.box.mr:.4f}")
    print(f"mAP50:     {metrics.box.map50:.4f}")
    print(f"mAP50-95:  {metrics.box.map:.4f}")


def train(data_yaml):
    """Train RT-DETR on the custom vehicle dataset (uses Ultralytics' default runs/detect/ output folder)."""
    model = RTDETR(MODEL_SIZE)  # loads COCO-pretrained weights

    results = model.train(
        data=data_yaml,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH,
        device=DEVICE,
        patience=PATIENCE,
        amp=AMP,
        plots=True,
    )

    save_dir = results.save_dir  # wherever Ultralytics decided to save this run
    best_weights = str(save_dir / "weights" / "best.pt")
    print(f"\nTraining complete. Best weights saved to: {best_weights}")
    return best_weights


# ---- run training ----
resolved_yaml = build_resolved_data_yaml()
best_weights = train(resolved_yaml)

# Explicit validation pass on the val split, with precision/recall
val_model = RTDETR(best_weights)
val_metrics = val_model.val(data=resolved_yaml, device=DEVICE)
print_metrics(val_metrics, "Validation (val split)")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Resolved data.yaml written to: /kaggle/working/data_resolved.yaml
  train: /kaggle/input/datasets/nomanhossain4/green-dataset/Vehicles.v5i.yolov11/train/images
  val: /kaggle/input/datasets/nomanhossain4/green-dataset/Vehicles.v5i.yolov11/valid/images
  test: /kaggle/input/datasets/nomanhossain4/green-dataset/Vehicles.v5i.yolov11/test/images
Ultralytics 8.4.150 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0,

In [2]:
# =====================================================================
# CELL 2 — TESTING (RT-DETR)
# Run this AFTER Cell 1 in the same notebook session — it reuses
# `best_weights` and `resolved_yaml` defined there.
# =====================================================================
best_weights = "/kaggle/input/datasets/imahroy/re-detr/weights/best.pt"

import yaml
from ultralytics import RTDETR

model = RTDETR(best_weights)

# ---- Metrics on the held-out test split ----
test_metrics = model.val(data=resolved_yaml, split="test", device=DEVICE)

print("\n--- Test split metrics ---")
print(f"Precision: {test_metrics.box.mp:.4f}")
print(f"Recall:    {test_metrics.box.mr:.4f}")
print(f"mAP50:     {test_metrics.box.map50:.4f}")
print(f"mAP50-95:  {test_metrics.box.map:.4f}")

# Per-class precision/recall/mAP
print("\n--- Per-class ---")
names = test_metrics.names
for i, c in enumerate(test_metrics.ap_class_index):
    p, r, ap50, ap = test_metrics.box.class_result(i)
    print(f"{names[c]:<12} P: {p:.3f}  R: {r:.3f}  mAP50: {ap50:.3f}  mAP50-95: {ap:.3f}")

# ---- Save annotated predictions on the test images ----
with open(resolved_yaml) as f:
    cfg = yaml.safe_load(f)
test_images_dir = cfg["test"]

model.predict(
    source=test_images_dir,
    imgsz=IMG_SIZE,
    conf=0.25,
    device=DEVICE,
    save=True,
    verbose=False,   # suppress per-image "image 1/566 ... 1 Van, 16.1ms" lines
)

print("\nAnnotated test predictions saved (see the 'predict' folder under runs/detect/).")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


NameError: name 'resolved_yaml' is not defined